In [ ]:
%pip install -q ipylab

# Autostart

Autostart is a feature implemented using the [pluggy](https://pluggy.readthedocs.io/en/stable/index.html#pluggy) plugin system. The code associated with the entry point `ipylab_backend` will be called (imported) when `ipylab` is activated. `ipylab` will activate when Jupyterlab is started (provided `ipylab` is installed and enabled). 

There are no limitations to what can be done. But it is recommended to import on demand to minimise the time required to launch.  Some possibilities include:
* Create and register custom commands;
* Create launchers;
* Modify the appearance of Jupyterlab.

## Entry points

Add the following in your `pyproject.toml`

``` toml
[project.entry-points.ipylab_backend]
myproject = "myproject.pluginmodule"
```

In `my_module.autostart.py` write code that will be run once.

Example:

```python
# @ipylab_plugin.py

import asyncio

async def startup():
    import ipylab
    
    app = ipylab.JupyterFrontEnd()  
    await app.read_wait()
    #Do everything to startup

ipylab_plugin = None # Provide an empty object with the expected name.
asyncio.create_task(startup())
```

## Example creating a launcher

In [ ]:
# @my_module.autostart.py

import ipylab


async def create_app(path):
    # The code in this function is called in the new kernel.
    # Ensure imports are performed inside the function.
    import ipywidgets as ipw

    import ipylab

    panel = ipylab.Panel()
    panel.title.label = path
    panel.title.caption = panel.app.kernelId
    error_button = ipw.Button(
        description="Do an error",
        tooltip="An error dialog will pop up when this is clicked.\n"
        "The dialog demonstrates the use of the `on_frontend_error` plugin.",
    )
    error_button.on_click(lambda _: panel.app.execute_command("Not a command"))
    panel.children = [ipw.HTML(f"<h3>{path}</h3> Welcome to my app.<br> kernel id: {panel.app.kernelId}"), error_button]

    # Demonstrate usage of a plugin
    class IpylabPlugins:
        # Define plugins (see IpylabHookspec for available hooks)
        @ipylab.hookimpl
        def on_frontend_error(self, obj, error, content):  # noqa: ARG002
            panel.app.dialog.show_error_message("Error", str(error))

    # Register plugin for this kernel.
    ipylab.hookspecs.pm.register(IpylabPlugins())  # type: ignore

    disposable = await panel.addToShell()
    # Shutdown the kernel when the window is closed.
    disposable.observe(lambda _: panel.app.shutdown_kernel(), names="comm")


n = 0
app = ipylab.JupyterFrontEnd()


async def start_my_app(cwd):  # noqa: ARG001
    n = getattr(start_my_app, "callcount", 0)
    n += 1
    start_my_app.callcount = n
    path = f"my app {n}"
    launcher = (
        ipylab.DisposableConnection(id=app.current_widget_id) if app.current_widget_id.startswith("launcher") else None
    )

    main_area = await app.exec_eval(
        execute=create_app,
        evaluate={"path": f"'{path}'", "payload": "create_app(path)"},
        path=path,
    )
    if launcher:
        launcher.dispose()
    return main_area


async def register_commands():
    await app.commands.addPythonCommand(
        "start_my_app", execute=start_my_app, label="Start Custom App", icon_class="jp-PythonIcon", just_coro=True
    )
    await app.launcher.add_item("start_my_app", "Ipylab")
    return "done"


ipylab_plugin = None
t = app.to_task(register_commands())

In [ ]:
t.result()

In [ ]:
app = ipylab.JupyterFrontEnd()

In [ ]:
t = app.execute_command("launcher:create")

In [ ]:
cmd = ipylab.DisposableConnection(id="start_my_app")

In [ ]:
cmd.dispose()

In [ ]:
app.launcher.remove_item("start_my_app", "Ipylab")